https://docs.nvidia.com/bionemo-framework/1.10/notebooks/model_training_molmim.html

In [6]:
import os
from pathlib import Path
import pandas as pd
import warnings
from rdkit import Chem
from nemo.collections.common.tokenizers.regex_tokenizer import RegExTokenizer
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

bionemo_home = "/workspace/bionemo"
os.environ['BIONEMO_HOME'] = bionemo_home
os.chdir(bionemo_home)

# Specify dataset
dataset_name = 'chembl_35'
task = 'pretraining'

train_df = pd.read_parquet(f"{bionemo_home}/data/chembl_35_train.parquet", columns=["canonical_smiles"])
val_df = pd.read_parquet(f"{bionemo_home}/data/chembl_35_val.parquet", columns=["canonical_smiles"])
test_df = pd.read_parquet(f"{bionemo_home}/data/chembl_35_test.parquet", columns=["canonical_smiles"])

train_df.shape, val_df.shape, test_df.shape

ModuleNotFoundError: No module named 'nemo'

In [9]:
max_token_length = 254
# Note: the maximum token length generated from the smiles string should be 2 less than the max_seq_length specified in the model config.
# This is to account for the extra tokens <BOS> and <EOS>

def vocab_compliance_check(smiles: str, tokenizer: RegExTokenizer, max_token_length: int) -> bool:
    """Checks if the SMILES string only contains vocabulary in the tokenizer's vocabulary
    and if the token length is less than or equal to `max_token_length"""
    tokens = tokenizer.text_to_tokens(smiles)
    vocab_allowed = tokenizer.vocab.keys()
    return set(tokens).issubset(set(vocab_allowed)) and len(tokens) <= max_token_length

model_name = "molmim"
print(f"Filtering out molecules which are not present in the {model_name} tokenizer vocabulary or with max token length greater than {max_token_length}...")
tokenizer_path = bionemo_home + "/tokenizers/molecule/{model_name}/vocab/{model_name}.{extension}"
tokenizer = RegExTokenizer().load_tokenizer(regex_file=tokenizer_path.format(model_name=model_name, extension="model"), vocab_file=tokenizer_path.format(model_name=model_name, extension="vocab"))
train_df["vocab_compliant"] = train_df["canonical_smiles"].apply(lambda smi: vocab_compliance_check(smi, tokenizer, max_token_length))
# Select only molecules which are vocab compliant
filtered_train_df = train_df.loc[train_df['vocab_compliant']]
print(f"{len(train_df) - len(filtered_train_df)} molecules removed.")

Filtering out molecules which are not present in the molmim tokenizer vocabulary or with max token length greater than 254...
[NeMo I 2025-05-03 16:31:21 regex_tokenizer:240] Loading vocabulary from file = /workspace/bionemo/tokenizers/molecule/molmim/vocab/molmim.vocab
[NeMo I 2025-05-03 16:31:21 regex_tokenizer:254] Loading regex from file = /workspace/bionemo/tokenizers/molecule/molmim/vocab/molmim.model
8742 molecules removed.


In [10]:
val_df["vocab_compliant"] = val_df["canonical_smiles"].apply(lambda smi: vocab_compliance_check(smi, tokenizer, max_token_length))
# Select only molecules which are vocab compliant
filtered_val_df = val_df.loc[val_df['vocab_compliant']]
print(f"{len(val_df) - len(filtered_val_df)} molecules removed.")

1221 molecules removed.


In [11]:
test_df["vocab_compliant"] = test_df["canonical_smiles"].apply(lambda smi: vocab_compliance_check(smi, tokenizer, max_token_length))
# Select only molecules which are vocab compliant
filtered_test_df = test_df.loc[test_df['vocab_compliant']]
print(f"{len(test_df) - len(filtered_test_df)} molecules removed.")

2409 molecules removed.


In [12]:
for set_name in ["train", "val", "test"]:
    if set_name == "train":
        filtered_df = filtered_train_df
    elif set_name == "test":
        filtered_df = filtered_test_df
    else:
        filtered_df = filtered_val_df

    # Save the filtered DataFrame to a CSV file
    output_dir = os.path.join(bionemo_home, f"data/processed/{task}/{set_name}")
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    outfilename = os.path.join(output_dir, f"{set_name}_set.csv")
    filtered_df.to_csv(outfilename, index=False)
    print(f"Saved {set_name} set molecules to {outfilename}")

Saved train set molecules to /workspace/bionemo/data/processed/pretraining/train/train_set.csv
Saved val set molecules to /workspace/bionemo/data/processed/pretraining/val/val_set.csv
Saved test set molecules to /workspace/bionemo/data/processed/pretraining/test/test_set.csv


### Training

In [21]:
import wandb
wandb.login()

True

 ## pre-training

In [22]:
import os

model_path = "/workspace/bionemo/models/molmim_70m_24_3.nemo"
os.environ["PYTHONPATH"] = "/workspace/bionemo"

try:
    os.system("rm -rf data/data_index")
except:
    pass

config_name: str = "pretrain_small_canonicalized"
# the config should be defined in: /workspace/bionemo/examples/molecule/molmim/conf/*.yaml
command = f"""
cd {bionemo_home} && python examples/molecule/molmim/pretrain.py \
    do_training=True \
    do_testing=True \
    ++model.data.dataset_path="data/processed/{task}/" \
    ++model.data.dataset.train="train_set" \
    ++model.data.dataset.val="val_set" \
    ++model.data.dataset.test="test_set" \
    ++model.data.index_mapping_dir="data/data_index/" \
    ++model.data.data_impl_kwargs.csv_mmap.data_col=0 \
    ++model.dwnstr_task_validation.enabled=False \
    ++exp_manager.create_wandb_logger=True \
    ++exp_manager.resume_if_exists=False \
    --config-path=conf \
    --config-name={config_name} \
"""
print(command)


cd /workspace/bionemo && python examples/molecule/molmim/pretrain.py     do_training=True     do_testing=True     ++model.data.dataset_path="data/processed/pretraining/"     ++model.data.dataset.train="train_set"     ++model.data.dataset.val="val_set"     ++model.data.dataset.test="test_set"     ++model.data.index_mapping_dir="data/data_index/"     ++model.data.data_impl_kwargs.csv_mmap.data_col=0     ++model.dwnstr_task_validation.enabled=False     ++exp_manager.create_wandb_logger=True     ++exp_manager.resume_if_exists=False     --config-path=conf     --config-name=pretrain_small_canonicalized 


In [ ]:
import subprocess

model_path = "/workspace/bionemo/models/molmim_70m_24_3.nemo"
os.environ["PYTHONPATH"] = "/workspace/bionemo"
# Define the command as a multiline string

max_steps: int = 2
val_check_interval: int = max_steps // 2
batch_size: int = 32 # 1 step should effectively be one epoch, i.e. 2 shot learning
config_name: str = "pretrain_small_canonicalized"
# the config should be defined in: /workspace/bionemo/examples/molecule/molmim/conf/*.yaml
command = f"""
cd {bionemo_home} && python examples/molecule/molmim/pretrain.py \
    do_training=True \
    do_testing=True \
    ++model.data.dataset_path="data/processed/experts_1/" \
    ++model.data.dataset.train="train_set" \
    ++model.data.dataset.val="val_set" \
    ++model.data.dataset.test="test_set" \
    ++model.data.index_mapping_dir="data/data_index/" \
    ++model.data.data_impl_kwargs.csv_mmap.data_col=0 \
    ++model.dwnstr_task_validation.enabled=False \
    ++model.global_batch_size={batch_size} \
    ++trainer.devices=1 \
    ++trainer.accelerator='gpu' \
    ++trainer.max_steps={max_steps} \
    ++trainer.val_check_interval={val_check_interval} \
    ++exp_manager.create_wandb_logger=True \
    ++exp_manager.resume_if_exists=True \
    --config-path=conf \
    --config-name={config_name}
"""

# Run the command using subprocess
result = subprocess.run(command, shell=True, capture_output=True, text=True)

# Print the output and error (if any)
print(result.stdout)
print("===================================")
print(result.stderr)

In [ ]:
# Copy the latest trained model to the model directory
base_path = "/result/nemo_experiments/MolMIM/"
checkpoint_path = os.path.join(base_path, f"MolMIM-{config_name.split('_')[1]}_pretraining", "checkpoints", "MolMIM.nemo")
print(f"Checkpoint path: {checkpoint_path}, is file: {os.path.isfile(checkpoint_path)}")

os.makedirs(os.path.join(bionemo_home, "data", "models"), exist_ok=True)

os.system(f"mv {checkpoint_path} {bionemo_home}/data/models/MolMIM_{config_name.split('_')[1]}_{task}_max_steps_{max_steps}.nemo")